## nb30 — Citation & Publication Lift by CORE Tier (RQ3)

Are the lift effects consistent across venue tiers, or concentrated at A* conferences?

**Data sources:**
- `data/matched/author_lift.csv` — has author_id, conference, lift_citations, lift_works
- `data/matched/conference_core_ranks.csv` — CORE rankings per conference acronym

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

MATCH   = Path('..') / 'data' / 'matched'
FIG_DIR = Path('..') / 'data' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
lift = pd.read_csv(MATCH / 'author_lift.csv')
core = pd.read_csv(MATCH / 'conference_core_ranks.csv')

# Normalise for merge
lift['conference'] = lift['conference'].str.strip().str.upper()
core['conference'] = core['conference'].str.strip().str.upper()

# Drop old core_rank if already present, then re-merge cleanly
if 'core_rank' in lift.columns:
    lift = lift.drop(columns='core_rank')

df = lift.merge(core, on='conference', how='left')
df['core_rank'] = df['core_rank'].fillna('Unranked')

print(df['core_rank'].value_counts())
print(df.shape)

In [ ]:
# treatment==1 => award authors
award = df[df['treatment'] == 1].copy()

# Cap extreme outliers
award = award[award['lift_citations'] <= 20].copy()

tier_order = ['A*', 'A', 'Unranked']
award['core_rank'] = award['core_rank'].apply(lambda x: x if x in tier_order else 'Unranked')

print(award.groupby('core_rank')[['lift_citations', 'lift_works']].describe().round(3))

In [ ]:
# Kruskal-Wallis test across tiers
gc = [award[award['core_rank'] == t]['lift_citations'].dropna() for t in tier_order]
gw = [award[award['core_rank'] == t]['lift_works'].dropna() for t in tier_order]

kw_c = stats.kruskal(*gc)
kw_w = stats.kruskal(*gw)

print(f'Citation lift  — KW H={kw_c.statistic:.3f}, p={kw_c.pvalue:.4f}')
print(f'Publication lift — KW H={kw_w.statistic:.3f}, p={kw_w.pvalue:.4f}')
print()
print(award.groupby('core_rank')[['lift_citations', 'lift_works']].median().round(3))

In [ ]:
COLORS = {'A*': '#00696e', 'A': '#8b3a0f', 'Unranked': '#4a4a8a'}

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

for ax, metric, label in zip(
    axes,
    ['lift_citations', 'lift_works'],
    ['Citation Lift', 'Publication Lift']
):
    for pos, tier in enumerate(tier_order, 1):
        data = award[award['core_rank'] == tier][metric].dropna()
        ax.boxplot(
            data, positions=[pos], widths=0.5, patch_artist=True,
            boxprops=dict(facecolor=COLORS[tier], alpha=0.4),
            medianprops=dict(color='black', linewidth=2.5),
            whiskerprops=dict(color=COLORS[tier], linewidth=1.5),
            capprops=dict(color=COLORS[tier], linewidth=1.5),
            flierprops=dict(marker='', linestyle='none')
        )
        jitter = np.random.uniform(-0.15, 0.15, size=len(data))
        ax.scatter(pos + jitter, data, alpha=0.2, s=8, color=COLORS[tier], zorder=2)
        med = data.median()
        ax.annotate(f'{med:.2f}\n(n={len(data)})', xy=(pos, med),
                    xytext=(pos + 0.3, med), fontsize=8, color=COLORS[tier], va='center')

    ax.axhline(1.0, color='gray', linestyle='dashed', linewidth=1, label='No lift (=1)')
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(tier_order, fontsize=11)
    ax.set_ylabel(label, fontsize=11)
    ax.set_title(f'{label} by CORE Tier\n(Award Authors)', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / '30_lift_by_core_tier.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 30_lift_by_core_tier.png')